# 04 – Dashboard Visualizations

This notebook puts together all the visual insights from the cleaned and engineered dataset.  
The goal is to give a clear, high-level picture of ratings, sentiment, trends, and product performance.


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from snowflake.snowpark.context import get_active_session

session = get_active_session()
sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (12,6)


In [ ]:
import numpy as np

sent_df['SENTIMENT_LABEL'] = np.where(
    sent_df['SENTIMENT_SCORE'] > 0, 'Positive',
    np.where(sent_df['SENTIMENT_SCORE'] < 0, 'Negative', 'Neutral')
)


### Loading the engineered feature dataset
Pulling the final table from Snowflake so we can use it for all charts below.


In [ ]:
session.create_dataframe(sent_df).write.save_as_table(
    "ANALYTICS.FEATURE_ENGINEERED_REVIEWS",
    mode="overwrite"
)


### Quick preview
Just checking the first few rows to confirm everything looks good before plotting.


In [ ]:
df = session.table("ANALYTICS.REVIEWS_CLEAN").to_pandas()
df.head()


### Rating distribution
Shows how many reviews exist for each rating score. Helps understand overall customer satisfaction levels.


In [ ]:
plt.figure()
sns.countplot(x="RATING", hue="RATING", data=df, palette="Blues", legend=False)
plt.title("Distribution of Review Ratings")
plt.xlabel("Rating")
plt.ylabel("Count")
plt.show()


### Review length comparison by rating
Looking at how long reviews tend to be for each rating. Sometimes low ratings come with longer explanations.


In [ ]:
df["REVIEW_LENGTH"] = df["REVIEW_TEXT"].str.len()

plt.figure(figsize=(12,6))
sns.boxplot(x="RATING", y="REVIEW_LENGTH", data=df)
plt.title("Review Length by Rating")
plt.ylabel("Length (Characters)")
plt.xlabel("Rating")
plt.show()


### Helpfulness vs rating
Checking whether higher-rated reviews tend to be marked as more helpful by other users.


In [ ]:
plt.figure(figsize=(10,5))
sns.barplot(
    x="RATING",
    y="HELPFUL_RATIO",
    hue="RATING",
    data=df,
    palette="Blues_d",
    legend=False
)
plt.title("Average Helpfulness Ratio by Rating")
plt.xlabel("Rating")
plt.ylabel("Helpfulness Ratio")
plt.show()


In [ ]:
sent_df = session.table("ANALYTICS.FEATURE_ENGINEERED_REVIEWS").to_pandas()
print(sent_df.columns.tolist())


### Overall sentiment breakdown
Shows how many reviews fall into positive, negative, and neutral sentiment groups.


In [ ]:
plt.figure(figsize=(10, 6))
sns.countplot(
    x="SENTIMENT_LABEL",
    data=sent_df,
    hue="SENTIMENT_LABEL",   
    palette="pastel",
    legend=False             
)
plt.title("Overall Sentiment Distribution")
plt.xlabel("Sentiment")
plt.ylabel("Count")
plt.show()


### Sentiment distribution across ratings
This shows how sentiment categories map to each rating score. Helps validate whether the sentiment scoring matches real user feedback.


In [ ]:
sent_percent = (
    sent_df.groupby(["RATING", "SENTIMENT_LABEL"])
           .size()
           .reset_index(name="COUNT")
)

sent_percent["PERCENT"] = (
    sent_percent["COUNT"] /
    sent_percent.groupby("RATING")["COUNT"].transform("sum") * 100
)

plt.figure(figsize=(12,6))
sns.barplot(
    x="RATING",
    y="PERCENT",
    hue="SENTIMENT_LABEL",
    data=sent_percent,
    palette="pastel"
)
plt.title("Sentiment Distribution Across Ratings (%)")
plt.ylabel("Percent")
plt.xlabel("Rating")
plt.legend(title="Sentiment")
plt.show()


### Feature correlation heatmap
Visualizing how different numerical features relate to each other (length, word count, sentiment score, rating, etc.).


In [ ]:
num_cols = [
    "RATING",
    "TEXT_LENGTH",
    "WORD_COUNT",
    "CAPS_RATIO",
    "EXCLAMATION_COUNT",
    "POSITIVE_WORDS",
    "NEGATIVE_WORDS",
    "SENTIMENT_SCORE",
]

corr = sent_df[num_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f")
plt.title("Correlation Between Rating, Sentiment, and Text Features")
plt.tight_layout()
plt.show()


### Top-rated products (with enough reviews)
Listing products with the highest average rating, only including items with a meaningful number of reviews.


In [ ]:
top_products = session.sql("""
    SELECT
        f.PRODUCT_ID,
        MIN(r.SUMMARY) AS PRODUCT_NAME,
        AVG(f.SENTIMENT_SCORE) AS AVG_SENTIMENT,
        COUNT(*) AS N_REVIEWS
    FROM ANALYTICS.FEATURE_ENGINEERED_REVIEWS f
    JOIN ANALYTICS.REVIEWS_CLEAN r
      ON f.REVIEW_ID = r.REVIEW_ID
    GROUP BY f.PRODUCT_ID
    HAVING COUNT(*) >= 50
    ORDER BY AVG_SENTIMENT DESC, N_REVIEWS DESC
    LIMIT 10
""").to_pandas()

top_products


### Highest-rated products
Same idea as before but shown visually to make the comparison easier.


In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(
    x="AVG_SENTIMENT",
    y="PRODUCT_NAME",
    data=top_products
)
plt.xlabel("Average Sentiment Score")
plt.ylabel("Product")
plt.title("Top 10 Products by Average Sentiment (≥ 50 Reviews)")
plt.tight_layout()
plt.show()


### Rating trend over time
Shows how average ratings change across dates. Helps identify shifts in product quality or customer expectations.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

reviews_time_df = session.sql("""
    SELECT
        REVIEW_DATE,
        COUNT(*) AS N_REVIEWS,
        AVG(RATING) AS AVG_RATING
    FROM ANALYTICS.REVIEWS_CLEAN
    GROUP BY REVIEW_DATE
    ORDER BY REVIEW_DATE
""").to_pandas()

plt.figure(figsize=(12,5))
sns.lineplot(x="REVIEW_DATE", y="N_REVIEWS", data=reviews_time_df)
plt.title("Daily Review Volume Over Time")
plt.xlabel("Date")
plt.ylabel("Number of Reviews")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


### Review length distribution
A simple look at how long reviews generally are, checking for patterns or outliers.


In [ ]:
feat_df = session.table("ANALYTICS.FEATURE_ENGINEERED_REVIEWS").to_pandas()

plt.figure(figsize=(10,6))
sns.boxplot(
    x="RATING",
    y="SENTIMENT_SCORE",
    data=feat_df
)
plt.title("Sentiment Score Distribution by Rating")
plt.xlabel("Rating")
plt.ylabel("Sentiment Score")
plt.tight_layout()
plt.show()
